# Read labels
- This is some code to read the output of the label tool and plot the data with the associated maxima and minima.

In [1]:
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from helpers.df_ops import prepare_df

LABELS_PATH = Path("labels.json")
DATA_DIR    = PROJECT_ROOT / "Data"
TOL         = 6

def find_file(fname):
    matches = list(DATA_DIR.rglob(fname))
    if not matches:
        raise FileNotFoundError(f"{fname} not found under {DATA_DIR}")
    return matches[0]

def load_raw(fname):
    raw_df = pd.read_csv(find_file(fname), sep=r'\s+', skip_blank_lines=True)
    try:
        raw_data_df = prepare_df(raw_df, add_prefix=False, relative=True)
    except Exception:
        raw_data_df = prepare_df(raw_df, add_prefix=True, relative=True)
    day_offset = raw_data_df['day'].min()
    raw_data_df = raw_data_df.copy()
    raw_data_df['day'] -= day_offset
    return raw_data_df, day_offset

with open(LABELS_PATH) as f:
    labels = json.load(f)

print(f"Loaded labels for {len(labels)} files:")
for fname, lbl in labels.items():
    tags = ''.join(
        f'  [{t.upper()}]'
        for t in ('const', 'bad')
        if lbl.get(t, False)
    )
    print(f"  {fname}: {len(lbl['maxima'])} maxima, {len(lbl['minima'])} minima{tags}")

Loaded labels for 60 files:
  HD160346_Mt_wilson_data.txt: 4 maxima, 4 minima
  HD201091_Mt_wilson_data.txt: 3 maxima, 4 minima
  HD81809_Mt_wilson_data.txt: 3 maxima, 3 minima
  hd10072_caii.txt: 1 maxima, 0 minima
  hd10700_caii.txt: 0 maxima, 0 minima  [CONST]
  hd10780_caii.txt: 2 maxima, 2 minima
  hd11131_caii.txt: 0 maxima, 0 minima  [BAD]
  hd12235_caii.txt: 0 maxima, 0 minima  [CONST]
  hd123_caii.txt: 0 maxima, 0 minima  [BAD]
  hd12953_caii.txt: 0 maxima, 0 minima  [BAD]
  hd13043_caii.txt: 0 maxima, 0 minima  [BAD]
  hd13421_caii.txt: 0 maxima, 0 minima  [CONST]
  hd1388_caii.txt: 0 maxima, 0 minima  [BAD]
  hd1461_caii.txt: 0 maxima, 0 minima  [BAD]
  hd16160_caii.txt: 2 maxima, 2 minima
  hd16673_caii.txt: 0 maxima, 0 minima  [CONST]
  hd166_caii.txt: 0 maxima, 0 minima  [BAD]
  hd17361_caii.txt: 0 maxima, 0 minima  [BAD]
  hd17925_caii.txt: 0 maxima, 1 minima
  hd18256_caii.txt: 0 maxima, 0 minima  [CONST]
  hd1835_caii.txt: 0 maxima, 0 minima  [CONST]
  hd19787_caii.txt

In [2]:
# # Migrate labels.json to offset-corrected coordinates (run once).
# # Skips files already marked as migrated.

# migrated_any = False

# for fname, lbl in labels.items():
#     if lbl.get('_offset_applied'):
#         continue
#     _, day_offset = load_raw(fname)
#     if day_offset != 0:
#         lbl['maxima'] = [v - day_offset for v in lbl['maxima']]
#         lbl['minima'] = [v - day_offset for v in lbl['minima']]
#         print(f"  {fname}: subtracted offset {day_offset:.4f}")
#     lbl['_offset_applied'] = True
#     migrated_any = True

# if migrated_any:
#     with open(LABELS_PATH, 'w') as f:
#         json.dump(labels, f, indent=2)
#     print("labels.json updated.")
# else:
#     print("All files already migrated — nothing to do.")

In [3]:
def load_clean(fname):
    raw_data_df, _ = load_raw(fname)
    med = raw_data_df['sind'].median()
    mad = (raw_data_df['sind'] - med).abs().median()
    return (
        raw_data_df[(raw_data_df['sind'] - med).abs() < TOL * mad]
        .sort_values('day')
        .reset_index(drop=True)
    )

def gaussian_smooth(t, y, sigma, n_eval=500):
    t_eval = np.linspace(t.min(), t.max(), n_eval)
    w = np.exp(-0.5 * ((t[:, None] - t_eval[None, :]) / sigma) ** 2)
    return t_eval, (w * y[:, None]).sum(axis=0) / w.sum(axis=0)

import numpy as np

fig, axes = plt.subplots(len(labels), 1, figsize=(20, 5 * len(labels)))
if len(labels) == 1:
    axes = [axes]

for ax, (fname, lbl) in zip(axes, labels.items()):
    try:
        data_df = load_clean(fname)
    except Exception as e:
        ax.text(0.5, 0.5, f"Could not load:\n{e}", transform=ax.transAxes,
                ha='center', va='center', color='red', fontsize=10)
        ax.set_title(fname)
        continue

    is_const = lbl.get('const', False)
    is_bad   = lbl.get('bad',   False)

    color = 'steelblue' if not is_bad else 'salmon'
    ax.plot(data_df['day'], data_df['sind'], '.', color=color,
            markersize=3, alpha=0.7, rasterized=True)

    # Gaussian-weighted moving average
    span  = data_df['day'].max() - data_df['day'].min()
    t_s, y_s = gaussian_smooth(data_df['day'].values, data_df['sind'].values, sigma=span * 0.04)
    ax.plot(t_s, y_s, '-', color='black', linewidth=1.5, alpha=0.6, zorder=3)

    if is_bad:
        ax.set_facecolor('#fff0f0')
        ax.text(0.5, 0.5, 'BAD DATA', transform=ax.transAxes,
                fontsize=28, color='red', alpha=0.3,
                ha='center', va='center', fontweight='bold')
    elif is_const:
        ax.set_facecolor('#fff8e1')
        ax.text(0.5, 0.5, 'CONST', transform=ax.transAxes,
                fontsize=28, color='#ffb300', alpha=0.4,
                ha='center', va='center', fontweight='bold')
    else:
        y_max = data_df['sind'].max()
        y_min = data_df['sind'].min()
        y_pad = (y_max - y_min) * 0.03

        for day in lbl['maxima']:
            ax.axvline(day, color='royalblue', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.text(day, y_max + y_pad, 'MAX', color='royalblue',
                    fontsize=7, ha='center', va='bottom', rotation=90)

        for day in lbl['minima']:
            ax.axvline(day, color='darkorange', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.text(day, y_min - y_pad, 'MIN', color='darkorange',
                    fontsize=7, ha='center', va='top', rotation=90)

    tags = ''.join(f'  [{t.upper()}]' for t in ('const', 'bad') if lbl.get(t, False))
    ax.set_title(f"{fname}{tags}")
    ax.set_xlabel("Days since first observation")
    ax.set_ylabel("S-index")

plt.tight_layout()
plt.show()